In [1]:
import os
import zipfile
import random
import shutil
from google.colab import drive
from PIL import Image

In [2]:
# --- 1. Mount Google Drive ---
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# --- 2. Define Paths ---
# *Update these paths to the actual locations in your Google Drive*
raw_zip_path = '/content/drive/MyDrive/Betel_Leaf_Raw.zip'        # Raw images zip
aug_zip_path = '/content/drive/MyDrive/Betel_Leaf_Augmented.zip'  # Augmented images zip
extracted_raw_folder = '/content/extracted_raw'
extracted_aug_folder = '/content/extracted_aug'

In [4]:
# --- Paths within the final dataset directory ---
final_dataset_dir = '/content/Betel_Leaf_Final_Dataset'
train_dir = os.path.join(final_dataset_dir, 'train')
test_dir = os.path.join(final_dataset_dir, 'test')
val_dir = os.path.join(final_dataset_dir, 'valid')
base_raw_dir = '/content/Betel_Leaf_Raw'       #where extracted raw data will be moved
base_aug_dir = '/content/Betel_Leaf_Augmented' #where extracted aug data will be moved

In [5]:
# --- Path for the final zipped dataset on Google Drive ---
output_zip_path = '/content/drive/MyDrive/Betel_Leaf_Final_Dataset.zip'  # Example. CHANGE!

In [6]:
# --- 3. Unzip Datasets ---
with zipfile.ZipFile(raw_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_raw_folder)

with zipfile.ZipFile(aug_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_aug_folder)

In [7]:
# --- 3.5 Move to /content ---
# Move raw
extracted_raw_main_folder = os.path.join(extracted_raw_folder, 'Betel_Leaf_Raw')
if os.path.exists(extracted_raw_main_folder):
  for item in os.listdir(extracted_raw_main_folder):
    s = os.path.join(extracted_raw_main_folder, item)
    d = os.path.join(base_raw_dir, item)
    if os.path.isdir(s):
      shutil.move(s, d) #move extracted folder to /content
    else:
      shutil.copy2(s, d) #it can files too, copy those
else:
    print(f"Error:  Expected folder '{extracted_raw_main_folder}' not found.")
    exit()

In [8]:
# Move aug
extracted_aug_main_folder = os.path.join(extracted_aug_folder, 'Betel_Leaf_Augmented')
if os.path.exists(extracted_aug_main_folder):
  for item in os.listdir(extracted_aug_main_folder):
    s = os.path.join(extracted_aug_main_folder, item)
    d = os.path.join(base_aug_dir, item)
    if os.path.isdir(s):
      shutil.move(s, d) #move extracted folder to /content
    else:
      shutil.copy2(s, d) #it can files too, copy those
else:
    print(f"Error:  Expected folder '{extracted_aug_main_folder}' not found.")
    exit()

In [9]:
# --- 4. Create Directories ---
os.makedirs(final_dataset_dir, exist_ok=True)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

In [10]:
# --- 5. Get Class Names (Should be the same for both datasets) ---
class_names = [folder_name for folder_name in os.listdir(base_raw_dir)
               if os.path.isdir(os.path.join(base_raw_dir, folder_name))]
print(f"Found classes: {class_names}")

Found classes: ['Fungal Brown Spot Disease', 'Dried Leaf', 'Healthy Leaf', 'Bacterial Leaf Disease']


In [11]:
# --- 6. Merge, Resize, and Split Data ---
train_ratio = 0.7
test_ratio = 0.15
val_ratio = 0.15

In [12]:
for class_name in class_names:
    # Create class-specific subdirectories in train, test, and val.
    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)

    # Get raw image paths.
    raw_class_dir = os.path.join(base_raw_dir, class_name)
    raw_image_paths = [os.path.join(raw_class_dir, img) for img in os.listdir(raw_class_dir)
                       if os.path.isfile(os.path.join(raw_class_dir, img))]

    # Get augmented image paths.
    aug_class_dir = os.path.join(base_aug_dir, class_name)
    aug_image_paths = [os.path.join(aug_class_dir, img) for img in os.listdir(aug_class_dir)
                       if os.path.isfile(os.path.join(aug_class_dir, img))]

    # Combine raw and augmented paths (but keep them separate for now).
    all_image_paths = raw_image_paths + aug_image_paths


    # Resize and copy images
    resized_image_paths = []  # Store paths *after* resizing
    for img_path in all_image_paths:
      try:
        img = Image.open(img_path)
        img = img.resize((256, 256))

        # Get the filename and create new path in temp location
        filename = os.path.basename(img_path)
        temp_save_path = os.path.join("/content/temp", filename)
        os.makedirs("/content/temp", exist_ok=True) #make /content/temp if not exist
        img.save(temp_save_path)  # Save resized image to /content/temp
        resized_image_paths.append(temp_save_path)

      except (IOError, OSError) as e:
        print(f"Error processing image {img_path}: {e}")
        # Decide how to handle: skip, delete the file, etc.
        continue  # Skip to the next image

    # Shuffle ALL paths (raw and augmented together) AFTER resizing
    random.shuffle(resized_image_paths)

    # Split into train, test, val.
    num_images = len(resized_image_paths)
    train_split = int(num_images * train_ratio)
    test_split = int(num_images * (train_ratio + test_ratio))

    train_images = resized_image_paths[:train_split]
    test_images = resized_image_paths[train_split:test_split]
    val_images = resized_image_paths[test_split:]


    # Copy images to their respective directories.
    for img_path in train_images:
        shutil.copy(img_path, os.path.join(train_dir, class_name))
    for img_path in test_images:
        shutil.copy(img_path, os.path.join(test_dir, class_name))
    for img_path in val_images:
        shutil.copy(img_path, os.path.join(val_dir, class_name))

    print(f"Class '{class_name}' split: Train={len(train_images)}, Test={len(test_images)}, Val={len(val_images)}")

Class 'Fungal Brown Spot Disease' split: Train=700, Test=150, Val=150
Class 'Dried Leaf' split: Train=700, Test=150, Val=150
Class 'Healthy Leaf' split: Train=700, Test=150, Val=150
Class 'Bacterial Leaf Disease' split: Train=700, Test=150, Val=150


In [16]:
# --- 7.  (Optional) Cleanup Extracted Folders ---
shutil.rmtree(extracted_raw_folder)
shutil.rmtree(extracted_aug_folder)
shutil.rmtree(base_raw_dir)
shutil.rmtree(base_aug_dir)
shutil.rmtree("/content/temp") #cleanup resized images

In [17]:
# --- 8. Zip the Final Dataset and Save to Google Drive ---
def zipdir(path, ziph):
    for root, dirs, files in os.walk(path):
        for file in files:
            rel_path = os.path.relpath(os.path.join(root, file), os.path.join(path, '..'))
            ziph.write(os.path.join(root, file), rel_path)

with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipdir(final_dataset_dir, zipf)

In [18]:
# --- 9. (Optional) Cleanup Final Dataset Folder ---
shutil.rmtree(final_dataset_dir)

print(f"Dataset merging, resizing, organization, and zipping complete! Zipped file saved to: {output_zip_path}")

Dataset merging, resizing, organization, and zipping complete! Zipped file saved to: /content/drive/MyDrive/Betel_Leaf_Final_Dataset.zip
